In [1]:
import pandas as pd
import requests
import cloudscraper
import io


# URL of the page containing the table
url = "https://www.visualcapitalist.com/ranked-top-buyers-u-s-oil-2025/"

# The User-Agent header is required so that the website does not block the request from the script
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}
dir = "./"

In [4]:
try:
    # Create scraper object
    scraper = cloudscraper.create_scraper(
        browser={
            'browser': 'chrome',
            'platform': 'windows',
            'desktop': True
        }
    )    
    print("Connecting to the website...")
    response = scraper.get(url)
    
    if response.status_code == 200:
        html_data = io.StringIO(response.text)
        tables = pd.read_html(html_data)
        
        if tables:
            # Select the first table
            df = tables[0]
            
            # 1. Convert "Share of Total" column to numeric
            if 'Share of Total' in df.columns:
                # Cleaning: remove %, replace commas with dots
                df['Share of Total'] = df['Share of Total'].astype(str).str.replace('%', '', regex=False)
                df['Share of Total'] = pd.to_numeric(df['Share of Total'].str.replace(',', '.', regex=False)) / 100
                
                # 2. Calculate share for "Rest of World" (1 minus sum of others)
                rest_share = 1 - df['Share of Total'].sum()
                
                # 3. Create a new row
                new_row = pd.DataFrame([{
                    'Rank': 31,
                    'Country': 'Rest of World',
                    'Region': 'World',
                    'Total Imports (Millions of Barrels 2025)': 424,
                    'Share of Total': rest_share
                }])
                
                # 4. Append the new row to the end of the table
                df = pd.concat([df, new_row], ignore_index=True)
            
            # Save to CSV
            filename = dir + "top_buyers_us_oil_2025.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            
            print(f"Success! Table saved to: {filename}")
            print("\nLast 5 rows of the table:")
            print(df.tail())
        else:
            print("No tables found on the page.")
    else:
        print(f"Access denied. Status code: {response.status_code}")

except Exception as e:
    print(f"An error occurred: {e}")


Connecting to the website...
Success! Table saved to: ./top_buyers_us_oil_2025.csv

Last 5 rows of the table:
    Rank        Country                   Region  \
26    27     🇧🇪 Belgium                   Europe   
27    28     🇲🇦 Morocco                   Africa   
28    29    🇭🇳 Honduras  Central & South America   
29    30      🇳🇴 Norway                   Europe   
30    31  Rest of World                    World   

    Total Imports (Millions of Barrels 2025)  Share of Total  
26                                        41           0.010  
27                                        38           0.010  
28                                        30           0.008  
29                                        30           0.008  
30                                       424           0.106  
